## Imports

Here import all crucial packages etc.

In [1]:
import json
import os
import pandas as pd
import torch
from sklearn.metrics import precision_score, recall_score, f1_score

In [2]:
from transformers import (
    Trainer,
    TrainingArguments,
    AutoTokenizer,
    AutoModelForSequenceClassification,
    pipeline,
    EvalPrediction
)

## Utils

Helper functions that you will use

In [3]:
#Code here
os.environ["WANDB_DISABLED"] = "true"

In [4]:
class DisinformationDataset(torch.utils.data.Dataset):
    """
    This class wraps our tokenized data and labels so PyTorch can easily loop through them during training. It converts each input into tensors and returns them with the label — all in the format the model expects.
    """
    # When we create an instance of dataset, we pass in encodings and labels
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    # This method tells PyTorch how to get one item (input + label).
    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    # Returns how many examples are in the dataset (needed by DataLoader).
    def __len__(self):
        return len(self.labels)


def load_and_process_data(file_path: str, label_column: str = "label") -> pd.DataFrame:
    """
    Loads the data from a CSV file and processes the labels.
    Args:
        file_path (str): Path to the CSV file.
        label_column (str): The column name containing the labels.
        text_column (str): The column name containing the text content.
    Returns:
        pd.DataFrame: Processed dataframe with labels and text content.
    """
    data = pd.read_csv(file_path, encoding='utf-8')
    data[label_column] = data[label_column].apply(lambda x: 1 if "fake" in x.lower() else 0)
    return data


def save_metrics_to_json(metrics: dict, output_file_path: str):
    """
    Saves the metrics to a JSON file.
    Args:
        metrics (dict): The evaluation metrics.
        output_file_path (str): The file path to save the metrics.
    """
    os.makedirs(os.path.dirname(output_file_path), exist_ok=True)
    with open(output_file_path, 'w') as output_file:
        json.dump(metrics, output_file, indent=4)

In [5]:
def compute_metrics(pred=None, y_true=None, y_pred=None):
    """
    Computes F1 scores (micro, macro, weighted) for both training and testing data.

    If `pred` is provided, it computes metrics for the trainer using `EvalPrediction`.
    If `y_true` and `y_pred` are provided, it computes metrics for test data predictions.

    Parameters:
        - pred (EvalPrediction, optional): The evaluation prediction object for Trainer.
        - y_true (list, optional): The ground truth labels for the test data.
        - y_pred (list, optional): The predicted labels for the test data.

    Returns:
        - dict: A dictionary containing F1 metrics.
    """
    if pred is not None:
        # When working with the Trainer, pred is an EvalPrediction object
        labels = pred.label_ids
        y_pred = pred.predictions.argmax(-1)
    elif y_true is not None and y_pred is not None:
        # If y_true and y_pred are provided, use them for test evaluation
        labels = y_true
    else:
        raise ValueError("Either `pred` or both `y_true` and `y_pred` must be provided.")

        # Compute F1 scores
    f1 = f1_score(y_true=labels, y_pred=y_pred)

    return {
        'f1': f1_score(labels, y_pred),
        'precision': precision_score(labels, y_pred),
        'recall': recall_score(labels, y_pred)
    }

def compute_metrics_for_trainer(pred: EvalPrediction):
    base_metrics = compute_metrics(pred=pred)
    return {
        "eval_f1": base_metrics["f1"],
        "eval_precision": base_metrics["precision"],
        "eval_recall": base_metrics["recall"]
    }

# Assignment

# Fine-Tuning BERT Model to Fake News detection

## Import Train, Validation and Test data

Import all datasets and load and preprocess train and validation

Link to direcotry with data: https://github.com/ArkadiusDS/NLP-Labs/tree/master/data/CoAID/

In [6]:
url_test = 'https://raw.githubusercontent.com/ArkadiusDS/NLP-Labs/refs/heads/master/data/CoAID/test.csv'
url_train = 'https://raw.githubusercontent.com/ArkadiusDS/NLP-Labs/refs/heads/master/data/CoAID/train.csv'
url_valid = 'https://raw.githubusercontent.com/ArkadiusDS/NLP-Labs/refs/heads/master/data/CoAID/validation.csv'

In [7]:
!wget -O test.csv {url_test}
!wget -O train.csv {url_train}
!wget -O validation.csv {url_valid}

--2025-05-21 20:08:28--  https://raw.githubusercontent.com/ArkadiusDS/NLP-Labs/refs/heads/master/data/CoAID/test.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.108.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 221757 (217K) [text/plain]
Saving to: ‘test.csv’

test.csv            100%[===================>] 216.56K  --.-KB/s    in 0.02s   

2025-05-21 20:08:29 (9.02 MB/s) - ‘test.csv’ saved [221757/221757]

--2025-05-21 20:08:29--  https://raw.githubusercontent.com/ArkadiusDS/NLP-Labs/refs/heads/master/data/CoAID/train.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 155653

In [8]:
train_data = load_and_process_data('train.csv')
validation_data = load_and_process_data('validation.csv')

## Load model and tokenizer

Firstly create two dicts id2label and label2id and then load model and tokenizer
Then use well-known distilled version of BERT model for faster fine-tuning: 'distilbert/distilbert-base-uncased' or any other model you wish.

In [9]:
id2label = {0: "Credible", 1: "Fake"}
label2id = {"Credible": 0, "Fake": 1}

In [10]:
# Load the pre-trained BERT model and tokenizer
# BERT is a transformer-based model that has been pre-trained on a large corpus of text
# We'll use it for classification task, where the model predicts labels for text.

# Load the BERT model for classification (the base uncased version of BERT) This
# is a generic model class that will be instantiated as one of the model classes
# of the library (with a sequence classification head) when created with the
# from_pretrained()
model = AutoModelForSequenceClassification.from_pretrained('google-bert/bert-base-uncased', num_labels=2, id2label=id2label, label2id=label2id)

# Load the corresponding tokenizer for BERT The tokenizer is responsible for
# converting the text into tokens that the model can process
tokenizer = AutoTokenizer.from_pretrained('google-bert/bert-base-uncased')

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at google-bert/bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


## Tokenize datasets and prepare it for fine-tuning

You may use DisinformationDataset class for data preparation.

In [11]:
# Tokenize the datasets (training and validation) to prepare them for input into the BERT model.
# Tokenization converts the raw text data into a format the BERT model can process.

# Tokenizing the training dataset
train_encodings = tokenizer(
        train_data['content'].tolist(),
        truncation=True,
        padding=True,
        max_length=256
    )

# Tokenizing the validation dataset
val_encodings = tokenizer(
        validation_data['content'].tolist(),
        truncation=True,
        padding=True,
        max_length=256
    )

In [12]:
# Create custom datasets for training and validation using the DisinformationDataset class.
# These datasets will format the tokenized text data and corresponding labels into a format that can be used by the model during training and evaluation.

# Create the training dataset: it combines the tokenized training data and corresponding labels
train_dataset = DisinformationDataset(train_encodings, train_data['label'].tolist())

# Create the validation dataset: it combines the tokenized validation data and corresponding labels
val_dataset = DisinformationDataset(val_encodings, validation_data['label'].tolist())# Create custom datasets for training and validation using the DisinformationDataset class.
# These datasets will format the tokenized text data and corresponding labels into a format that can be used by the model during training and evaluation.

# Create the training dataset: it combines the tokenized training data and corresponding labels
train_dataset = DisinformationDataset(train_encodings, train_data['label'].tolist())

# Create the validation dataset: it combines the tokenized validation data and corresponding labels
val_dataset = DisinformationDataset(val_encodings, validation_data['label'].tolist())

In [13]:
# https://huggingface.co/docs/transformers/v4.51.3/en/main_classes/trainer#transformers.TrainingArguments

In [14]:
def run_experiment(exp_id, learning_rate, num_train_epochs, weight_decay, warmup_ratio):
    model = AutoModelForSequenceClassification.from_pretrained(
        'google-bert/bert-base-uncased',
        num_labels=2,
        id2label={0: "Credible", 1: "Fake"},
        label2id={"Credible": 0, "Fake": 1}
    )

    args = TrainingArguments(
        output_dir=f'output/exp_{exp_id}/',
        learning_rate=learning_rate,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        num_train_epochs=num_train_epochs,
        warmup_ratio=warmup_ratio,
        weight_decay=weight_decay,
        fp16=True,
        metric_for_best_model='f1',
        load_best_model_at_end=True,
        save_total_limit=2,
        greater_is_better=True,
        save_strategy='steps',
        eval_strategy='steps',
        eval_steps=100,
        report_to=[]
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        compute_metrics=compute_metrics_for_trainer
    )

    trainer.train()

    metrics = trainer.evaluate()
    model.save_pretrained(f'output/exp_{exp_id}/')
    tokenizer.save_pretrained(f'output/exp_{exp_id}/')

    return {
        "model": "google-bert/bert-base-uncased",
        "hyperparameters": {
            "learning_rate": learning_rate,
            "num_train_epochs": num_train_epochs,
            "weight_decay": weight_decay,
            "warmup_ratio": warmup_ratio
        },
        "f1_score": metrics["eval_f1"],
        "precision": metrics["eval_precision"],
        "recall": metrics["eval_recall"]
    }


## Fine-tune BERT model on at least 3 sets of hyperparameters

Check F1 score, precision and recall for each fine-tuned model and at the end choose set of hyperparameters that gives you best results. For each set of hyperparameters write down the final metrics. You need to acheive at least below result on validation dataset:

"f1": 0.91,
"recall": 0.91,
"precision": 0.91

Remember you need to achieve these minimum results on VALIDATION dataset and the best model on validation dataset will have to be used for predictions on test dataset.


In [15]:
exp0 = run_experiment("exp0", learning_rate=2e-5, num_train_epochs=4, weight_decay=0.1, warmup_ratio=0.1)
exp1 = run_experiment("exp1", learning_rate=3e-5, num_train_epochs=3, weight_decay=0.05, warmup_ratio=0.06)
exp2 = run_experiment("exp2", learning_rate=1e-5, num_train_epochs=5, weight_decay=0.2, warmup_ratio=0.0)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at google-bert/bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Step,Training Loss,Validation Loss,F1,Precision,Recall
100,No log,0.290145,0.746177,1.000000,0.595122
200,No log,0.082049,0.938776,0.983957,0.897561
300,No log,0.113170,0.924675,0.988889,0.868293
400,No log,0.064423,0.962594,0.984694,0.941463
500,0.132800,0.068017,0.953317,0.960396,0.946341
600,0.132800,0.092714,0.944444,0.979058,0.912195
700,0.132800,0.082625,0.958025,0.970000,0.946341
800,0.132800,0.092783,0.950000,0.974359,0.926829


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at google-bert/bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Step,Training Loss,Validation Loss,F1,Precision,Recall
100,No log,0.138061,0.887097,0.988024,0.804878
200,No log,0.083606,0.940000,0.964103,0.917073
300,No log,0.173448,0.876712,1.000000,0.780488
400,No log,0.090483,0.949749,0.979275,0.921951
500,0.123000,0.070468,0.960199,0.979695,0.941463
600,0.123000,0.086275,0.946835,0.984211,0.912195


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at google-bert/bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Step,Training Loss,Validation Loss,F1,Precision,Recall
100,No log,0.223466,0.802326,0.992806,0.673171
200,No log,0.081508,0.947631,0.969388,0.926829
300,No log,0.100064,0.935065,1.000000,0.878049
400,No log,0.072494,0.954545,0.989529,0.921951
500,0.137500,0.057764,0.948148,0.960000,0.936585
600,0.137500,0.098646,0.940568,1.000000,0.887805
700,0.137500,0.094728,0.938776,0.983957,0.897561
800,0.137500,0.084765,0.947368,0.974227,0.921951
900,0.137500,0.078405,0.952618,0.974490,0.931707
1000,0.023300,0.070733,0.955224,0.974619,0.936585


In [16]:
# Save the trained model to a specified directory after training is completed.
# This allows you to persist the model and use it for future predictions or fine-tuning without retraining.
best_exp = max([exp0, exp1, exp2], key=lambda x: x["f1_score"])
exp_map = {"exp0": exp0, "exp1": exp1, "exp2": exp2}
best_exp_id = [k for k, v in exp_map.items() if v == best_exp][0]
best_hparams = best_exp["hyperparameters"]

# Mostrar cuál fue el mejor
print(f"Best experiment: {best_exp_id}")
print(f"Hyperparameters: {best_hparams}")

Best experiment: exp1
Hyperparameters: {'learning_rate': 3e-05, 'num_train_epochs': 3, 'weight_decay': 0.05, 'warmup_ratio': 0.06}


## Final prediction on test dataset

Take best model and hyperparameters on validation and predict on test dataset. Compute evaluation metrics f1, precision and recall.

In [17]:
# Entrenamiento del modelo final usando los hiperparámetros del mejor experimento
training_args = TrainingArguments(
    output_dir="output/final/",
    learning_rate=best_hparams["learning_rate"],
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=best_hparams["num_train_epochs"],
    warmup_ratio=best_hparams["warmup_ratio"],
    weight_decay=best_hparams["weight_decay"],
    fp16=True,
    eval_strategy="steps",
    save_strategy="steps",
    eval_steps=100,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    save_total_limit=2,
    greater_is_better=True,
    report_to=[]
)

In [18]:
model = AutoModelForSequenceClassification.from_pretrained(
    "google-bert/bert-base-uncased",
    num_labels=2,
    id2label={0: "Credible", 1: "Fake"},
    label2id={"Credible": 0, "Fake": 1}
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics_for_trainer
)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at google-bert/bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [19]:
trainer.train()

Step,Training Loss,Validation Loss,F1,Precision,Recall
100,No log,0.163619,0.874317,0.993789,0.780488
200,No log,0.077857,0.939698,0.968912,0.912195
300,No log,0.093740,0.932990,0.989071,0.882927
400,No log,0.083709,0.941772,0.978947,0.907317
500,0.132300,0.065308,0.950495,0.964824,0.936585
600,0.132300,0.076746,0.947631,0.969388,0.926829


TrainOutput(global_step=660, training_loss=0.10328744975003329, metrics={'train_runtime': 171.2441, 'train_samples_per_second': 61.439, 'train_steps_per_second': 3.854, 'total_flos': 1384095706721280.0, 'train_loss': 0.10328744975003329, 'epoch': 3.0})

In [20]:
# Guardar el modelo final
trainer.save_model("output/final/")
tokenizer.save_pretrained("output/final/")

('output/final/tokenizer_config.json',
 'output/final/special_tokens_map.json',
 'output/final/vocab.txt',
 'output/final/added_tokens.json',
 'output/final/tokenizer.json')

In [21]:
# Load the test data and preprocess
test_data = load_and_process_data('test.csv')

In [22]:
# Cargar el test
test_data = load_and_process_data('test.csv')

# Cargar modelo y tokenizer desde carpeta local
model = AutoModelForSequenceClassification.from_pretrained("output/final")
tokenizer = AutoTokenizer.from_pretrained("output/final")

classifier = pipeline(
    task="text-classification",
    model=model,
    tokenizer=tokenizer,
    device=0,  # o -1 si estás en CPU
    truncation=True,
    padding=True,
    max_length=256
)

# Predicción por lotes
results = classifier(test_data["content"].tolist(), batch_size=32)

# Convertir a etiquetas binarias
test_data["predictions"] = [1 if r["label"] == "Fake" else 0 for r in results]


Device set to use cuda:0


In [23]:
base_metrics = compute_metrics(
    y_true=test_data["label"],
    y_pred=test_data["predictions"]
)

evaluation_results = {
    "test_f1": base_metrics["f1"],
    "test_precision": base_metrics["precision"],
    "test_recall": base_metrics["recall"]
}

In [24]:
print(evaluation_results)

{'test_f1': 0.9705882352941176, 'test_precision': 0.9801980198019802, 'test_recall': 0.9611650485436893}


# Final file with results and description

In [25]:
import json
from datasets import Dataset

All keys in your dictionary have to be the same as below. The only changes you should do in terms of keys is changing names of hyperparameters, e.g. instead of key "name_of_hyperparameter_0" if you used learning rate then write "learning_rate". Other important information in the dictionary below and comments. Each value says what is expected.

Example dictionary provided under the template.

In [26]:
output_file_path = "metrics/results.json"
save_metrics_to_json(evaluation_results, output_file_path)

In [27]:
data = {
    "experiment_0": {
        "model": "google-bert/bert-base-uncased",
        "hyperparameters": exp0["hyperparameters"],
        "f1_score": exp0["f1_score"],
        "precision": exp0["precision"],
        "recall": exp0["recall"],
        "description": "This experiment fine-tuned the model using learning rate {}, weight decay {}, warmup ratio {}, and trained for {} epochs.".format(
            exp0["hyperparameters"]["learning_rate"],
            exp0["hyperparameters"]["weight_decay"],
            exp0["hyperparameters"]["warmup_ratio"],
            exp0["hyperparameters"]["num_train_epochs"]
        )
    },
    "experiment_1": {
        "model": "google-bert/bert-base-uncased",
        "hyperparameters": exp1["hyperparameters"],
        "f1_score": exp1["f1_score"],
        "precision": exp1["precision"],
        "recall": exp1["recall"],
        "description": "Alternative configuration with learning rate {}, trained for {} epochs. It showed a tradeoff between recall and precision.".format(
            exp1["hyperparameters"]["learning_rate"],
            exp1["hyperparameters"]["num_train_epochs"]
        )
    },
    "experiment_2": {
        "model": "google-bert/bert-base-uncased",
        "hyperparameters": exp2["hyperparameters"],
        "f1_score": exp2["f1_score"],
        "precision": exp2["precision"],
        "recall": exp2["recall"],
        "description": "This setup used more training epochs and higher weight decay. It achieved slightly lower F1 compared to the others."
    },
    "final_prediction": {
        "model": "google-bert/bert-base-uncased",
        "experiment_chosen": best_exp_id,
        "hyperparameters": best_hparams,
        "f1_score": evaluation_results["test_f1"],
        "precision": evaluation_results["test_precision"],
        "recall": evaluation_results["test_recall"],
        "description": "Final prediction using the best validation configuration ({}). Achieved F1 {:.4f}, precision {:.4f}, recall {:.4f} on test data.".format(
            best_exp_id,
            evaluation_results["test_f1"],
            evaluation_results["test_precision"],
            evaluation_results["test_recall"]
        )
    }
}

In [28]:
final_prediction = {
    "model": "google-bert/bert-base-uncased",
    "experiment_chosen": best_exp_id,
    "hyperparameters": best_hparams,
    "f1_score": evaluation_results["test_f1"],
    "precision": evaluation_results["test_precision"],
    "recall": evaluation_results["test_recall"],
    "description": "Final prediction on test set using best hyperparameters from validation. This model was fine-tuned using experiment settings from best_exp_id and shows strong generalization performance."
}

final_prediction = data["final_prediction"]

with open("experiments_Arkadiusz_Modzelewski_29580.json", "w") as f:
    json.dump(data, f, indent=4)

Template for your structured resulting file

In [29]:
# Template structure – not needed in final submission
'''data = {
    # Everything in experiment_0 is related to experiment on validation dataset, so metrics are computed on validation dataset etc.
    "experiment_0": {
        "model": "model name",
        "hyperparameters": {
            "learning_rate": "value in str or float - You need to play with at least two different hyperparameters so at least name_of_hyperparameter_0 and name_of_hyperparameter_1",
            "num_train_epochs": "value in str or float",
            "weight_decay": "value in str or float",
            "warmup_ratio": "value in str or float"
        },
        "f1_score": "value in float",
        "precision": "value in float",
        "recall": "value in float",
        "description": "Unique description one of the approach - it has to be different for each experiment."
    },
    # Everything in experiment_1 is related to experiment on validation dataset, so metrics are computed on validation dataset etc.
    "experiment_1": {
        "model": "model name",
        "hyperparameters": {
            "learning_rate": "value in str or float",
            "num_train_epochs": "value in str or float",
            "weight_decay": "value in str or float",
            "warmup_ratio": "value in str or float"
        },
        "f1_score": "value in float",
        "precision": "value in float",
        "recall": "value in float",
        "description": "Unique description two of the approach - it has to be different for each experiment."
    },
    # Everything in experiment_2 is related to experiment on validation dataset, so metrics are computed on validation dataset etc.
    "experiment_2": {
        "model": "model name",
        "hyperparameters": {
            "learning_rate": "value in str or float",
            "num_train_epochs": "value in str or float",
            "weight_decay": "value in str or float",
            "warmup_ratio": "value in str or float"
        },
        "f1_score": "value in float",
        "precision": "value in float",
        "recall": "value in float",
        "description": "Unique description three of the approach - it has to be different for each experiment."
    },
    # Everything in final_prediction is related to prediction on test dataset, so metrics are computed on test dataset etc.
    "final_prediction": {
        "model": "google-bert/bert-base-uncased",
        "experiment_chosen": "experiment_0 or experiment_1 or experiment_2",
        "hyperparameters": {
            "learning_rate": "value in str or float",
            "num_train_epochs": "value in str or float",
            "weight_decay": "value in str or float",
            "warmup_ratio": "value in str or float"
        },
        "f1_score": "value in float",
        "precision": "value in float",
        "recall": "value in float",
        "description": "Unique description four of the final results and prediction - it has to be different and here you will describe results on test dataset."
    }
}
'''

'data = {\n    # Everything in experiment_0 is related to experiment on validation dataset, so metrics are computed on validation dataset etc.\n    "experiment_0": {\n        "model": "model name",\n        "hyperparameters": {\n            "learning_rate": "value in str or float - You need to play with at least two different hyperparameters so at least name_of_hyperparameter_0 and name_of_hyperparameter_1",\n            "num_train_epochs": "value in str or float",\n            "weight_decay": "value in str or float",\n            "warmup_ratio": "value in str or float"\n        },\n        "f1_score": "value in float",\n        "precision": "value in float",\n        "recall": "value in float",\n        "description": "Unique description one of the approach - it has to be different for each experiment."\n    },\n    # Everything in experiment_1 is related to experiment on validation dataset, so metrics are computed on validation dataset etc.\n    "experiment_1": {\n        "model": "m

In [30]:
'''with open("experiments_name_surname_student_id.json", "w") as f:
    json.dump(data, f, indent=4)'''

'with open("experiments_name_surname_student_id.json", "w") as f:\n    json.dump(data, f, indent=4)'

## Example final file

In [31]:
# Map expX to experiment_X for JSON compatibility
experiment_name_map = {
    "exp0": "experiment_0",
    "exp1": "experiment_1",
    "exp2": "experiment_2"
}
best_exp_id = experiment_name_map[best_exp_id]

data = {
    "experiment_0": {
        "model": "google-bert/bert-base-uncased",
        "hyperparameters": exp0["hyperparameters"],
        "f1_score": exp0["f1_score"],
        "precision": exp0["precision"],
        "recall": exp0["recall"],
        "description": "This experiment fine-tuned the model using learning rate {}, weight decay {}, warmup ratio {}, and trained for {} epochs.".format(
            exp0["hyperparameters"]["learning_rate"],
            exp0["hyperparameters"]["weight_decay"],
            exp0["hyperparameters"]["warmup_ratio"],
            exp0["hyperparameters"]["num_train_epochs"]
        )
    },
    "experiment_1": {
        "model": "google-bert/bert-base-uncased",
        "hyperparameters": exp1["hyperparameters"],
        "f1_score": exp1["f1_score"],
        "precision": exp1["precision"],
        "recall": exp1["recall"],
        "description": "Alternative configuration with learning rate {}, trained for {} epochs. It showed a tradeoff between recall and precision.".format(
            exp1["hyperparameters"]["learning_rate"],
            exp1["hyperparameters"]["num_train_epochs"]
        )
    },
    "experiment_2": {
        "model": "google-bert/bert-base-uncased",
        "hyperparameters": exp2["hyperparameters"],
        "f1_score": exp2["f1_score"],
        "precision": exp2["precision"],
        "recall": exp2["recall"],
        "description": "This setup used more training epochs and higher weight decay. It achieved slightly lower F1 compared to the others."
    },
    "final_prediction": {
        "model": "google-bert/bert-base-uncased",
        "experiment_chosen": best_exp_id,
        "hyperparameters": best_hparams,
        "f1_score": evaluation_results["test_f1"],
        "precision": evaluation_results["test_precision"],
        "recall": evaluation_results["test_recall"],
        "description": "Final prediction using the best validation configuration ({}). Achieved F1 {:.4f}, precision {:.4f}, recall {:.4f} on test data.".format(
            best_exp_id,
            evaluation_results["test_f1"],
            evaluation_results["test_precision"],
            evaluation_results["test_recall"]
        )
    }
}

In [32]:
with open("experiments_Arkadiusz_Modzelewski_29580.json", "w") as f:
    json.dump(data, f, indent=4)

In [33]:
with open("experiments_Arkadiusz_Modzelewski_29580.json", "r") as f:
    contents = json.load(f)

print(json.dumps(contents, indent=4))

{
    "experiment_0": {
        "model": "google-bert/bert-base-uncased",
        "hyperparameters": {
            "learning_rate": 2e-05,
            "num_train_epochs": 4,
            "weight_decay": 0.1,
            "warmup_ratio": 0.1
        },
        "f1_score": 0.9502487562189055,
        "precision": 0.9695431472081218,
        "recall": 0.9317073170731708,
        "description": "This experiment fine-tuned the model using learning rate 2e-05, weight decay 0.1, warmup ratio 0.1, and trained for 4 epochs."
    },
    "experiment_1": {
        "model": "google-bert/bert-base-uncased",
        "hyperparameters": {
            "learning_rate": 3e-05,
            "num_train_epochs": 3,
            "weight_decay": 0.05,
            "warmup_ratio": 0.06
        },
        "f1_score": 0.9601990049751243,
        "precision": 0.9796954314720813,
        "recall": 0.9414634146341463,
        "description": "Alternative configuration with learning rate 3e-05, trained for 3 epochs. It sho

In [34]:
from google.colab import files
files.download("experiments_Arkadiusz_Modzelewski_29580.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>